In [1]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, insert, func
from sqlalchemy.orm import Session
import polars as pl

# Importing from 'app' module
from app.config import db_engine
from app.models import SilverCleanAd, GoldMarketTrend

In [2]:
with db_engine.connect() as connection:
    df_silver_raw = pl.read_database(
        select(
            SilverCleanAd.date,
            SilverCleanAd.category,
            SilverCleanAd.brand,
            SilverCleanAd.price,
            SilverCleanAd.ad_id
        ),
        connection=connection
    )

In [3]:
df_gold_market_trends = (
    df_silver_raw
    .group_by(['date', 'category', 'brand'])
    .agg(
        pl.col('price').median().alias('median_market_price'),
        pl.col('ad_id').count().alias('total_volume_available')
    )
)

In [4]:
if not df_gold_market_trends.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(GoldMarketTrend), df_gold_market_trends.to_dicts()
        )
else:
    print("No data found to insert.")